# AgroSight — Yield Prediction Model

**SP Jain | Group 3**

Synthetic dataset (domain-informed) → 4-model comparison → SHAP → export

**Run:** Cell 1 (install if needed) → Run All from Cell 2

In [ ]:
# CELL 1 — INSTALL
!pip install -q xgboost shap scikit-learn matplotlib seaborn pandas joblib
print("Ready — run Cell 2+")

In [ ]:
# CELL 2 — IMPORTS + PATHS
import warnings, json, joblib
from pathlib import Path
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import xgboost as xgb
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error, r2_score

np.random.seed(42)

ROOT       = Path.cwd()
ASSETS_DIR = ROOT / 'dashboard' / 'assets'
DATA_DIR   = ROOT / 'dashboard' / 'data'
MODELS_DIR = ROOT / 'models'
for d in [ASSETS_DIR, DATA_DIR, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

BG, CARD, GRID, TICK = '#0a0f1e', '#111827', '#1e293b', '#94a3b8'

def style_ax(ax):
    ax.set_facecolor(CARD)
    ax.tick_params(colors=TICK, labelsize=9)
    ax.xaxis.label.set_color(TICK)
    ax.yaxis.label.set_color(TICK)
    ax.title.set_color('#e2e8f0')
    for s in ax.spines.values():
        s.set_color(GRID)

FEATURES = [
    'moisture_pct', 'batch_weight_kg', 'ambient_temp_celsius',
    'processing_duration_min', 'machine_speed_rpm',
    'raw_material_grade', 'defect_rate_pct'
]
TARGET = 'yield_pct'
print('Paths OK:', ASSETS_DIR)

In [ ]:
# CELL 3 — GENERATE SYNTHETIC DATASET (5,000 rows)
N = 5000

df = pd.DataFrame({
    'moisture_pct':            np.random.uniform(8, 25, N),
    'batch_weight_kg':         np.random.uniform(10, 500, N),
    'ambient_temp_celsius':    np.random.uniform(18, 42, N),
    'processing_duration_min': np.random.uniform(15, 180, N),
    'machine_speed_rpm':       np.random.uniform(200, 1200, N),
    'raw_material_grade':      np.random.randint(1, 6, N),
    'defect_rate_pct':         np.random.uniform(0, 30, N),
})

noise = np.random.normal(0, 3, N)
df[TARGET] = (
    85
    - 0.8  * df['moisture_pct']
    - 0.4  * df['defect_rate_pct']
    + 2.5  * df['raw_material_grade']
    - 0.05 * np.abs(df['ambient_temp_celsius'] - 28)
    + 0.01 * df['processing_duration_min']
    + noise
).clip(40, 98)

print(df.describe().round(2))
print(f'\nRows: {len(df)} | Yield range: {df[TARGET].min():.1f}% – {df[TARGET].max():.1f}%')

In [ ]:
# CELL 4 — TRAIN / TEST SPLIT
X = df[FEATURES]
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {len(X_train)} | Test: {len(X_test)}')

In [ ]:
# CELL 5 — TRAIN 4 MODELS
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=200, random_state=42),
    'XGBoost': xgb.XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.05,
                                  random_state=42, verbosity=0),
}

stacked = StackingRegressor(
    estimators=[
        ('rf', RandomForestRegressor(n_estimators=100, random_state=42)),
        ('xgb', xgb.XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.05,
                                  random_state=42, verbosity=0)),
    ],
    final_estimator=LinearRegression(),
    cv=5
)
models['Stacked Ensemble'] = stacked

results = {}
predictions = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    predictions[name] = pred
    results[name] = {
        'MAPE': mean_absolute_percentage_error(y_test, pred) * 100,
        'RMSE': np.sqrt(mean_squared_error(y_test, pred)),
        'R2':   r2_score(y_test, pred),
    }
    print(f"{name:22s}  MAPE={results[name]['MAPE']:.2f}%  RMSE={results[name]['RMSE']:.3f}  R2={results[name]['R2']:.4f}")

metrics_df = pd.DataFrame(results).T.sort_values('R2', ascending=False)
BEST = metrics_df.index[0]
best_model = models[BEST]
print(f'\nWinner: {BEST}')

In [ ]:
# CELL 6 — METRICS COMPARISON CHART
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.patch.set_facecolor(BG)
colors = ['#6366f1', '#06b6d4', '#10b981', '#f59e0b']
names = metrics_df.index.tolist()

for ax, metric, title in zip(axes, ['MAPE', 'RMSE', 'R2'],
                              ['MAPE (%) — Lower', 'RMSE — Lower', 'R² — Higher']):
    style_ax(ax)
    vals = metrics_df[metric]
    bars = ax.bar(names, vals, color=colors, width=0.55)
    ax.set_title(title, fontweight='bold')
    ax.set_xticklabels(names, rotation=25, ha='right', fontsize=8)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f'{v:.2f}', ha='center', va='bottom', color='white', fontsize=8)

plt.tight_layout()
out = ASSETS_DIR / '04_yield_model_comparison.png'
plt.savefig(out, dpi=130, bbox_inches='tight', facecolor=BG)
plt.close(fig)
print(f'Saved: {out}')

In [ ]:
# CELL 7 — ACTUAL vs PREDICTED + RESIDUALS (winner model)
y_pred = predictions[BEST]
residuals = y_test.values - y_pred

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor(BG)

style_ax(ax1)
ax1.scatter(y_test, y_pred, alpha=0.5, color='#6366f1', s=20)
lims = [min(y_test.min(), y_pred.min())-2, max(y_test.max(), y_pred.max())+2]
ax1.plot(lims, lims, '--', color='#94a3b8', lw=1)
ax1.set_xlabel('Actual Yield (%)')
ax1.set_ylabel('Predicted Yield (%)')
ax1.set_title(f'{BEST} — Actual vs Predicted', fontweight='bold')

style_ax(ax2)
ax2.hist(residuals, bins=30, color='#06b6d4', edgecolor=GRID, alpha=0.85)
ax2.axvline(0, color='#f59e0b', lw=1.5)
ax2.set_xlabel('Residual (Actual − Predicted)')
ax2.set_title('Residuals Distribution', fontweight='bold')

plt.tight_layout()
out = ASSETS_DIR / '05_yield_predictions.png'
plt.savefig(out, dpi=130, bbox_inches='tight', facecolor=BG)
plt.close(fig)
print(f'Saved: {out}')

In [ ]:
# CELL 8 — SHAP FEATURE IMPORTANCE (XGBoost for explainability)
xgb_model = models['XGBoost']
explainer = shap.Explainer(xgb_model, X_train)
shap_vals = explainer(X_test.iloc[:200])

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_vals, X_test.iloc[:200], show=False)
fig = plt.gcf()
fig.patch.set_facecolor(BG)
for ax in fig.axes:
    ax.set_facecolor(CARD)
    ax.tick_params(colors=TICK)
plt.tight_layout()
out = ASSETS_DIR / '06_shap_importance.png'
plt.savefig(out, dpi=130, bbox_inches='tight', facecolor=BG)
plt.close(fig)
print(f'Saved: {out}')

In [ ]:
# CELL 9 — EXPORT MODEL + SIMULATOR LOOKUP TABLE
pkl_path = MODELS_DIR / 'agrosight_yield_model.pkl'
joblib.dump(best_model, pkl_path)
print(f'Saved model: {pkl_path}')

# Pre-computed lookup for dashboard scenario simulator
moisture_grid  = [10, 15, 20, 25]
speed_grid     = [400, 600, 800, 1000]
batch_grid     = [50, 150, 300]
grade_grid     = [1, 2, 3, 4, 5]
lookup = []
for m in moisture_grid:
    for s in speed_grid:
        for b in batch_grid:
            for g in grade_grid:
                row = {
                    'moisture_pct': m,
                    'batch_weight_kg': b,
                    'ambient_temp_celsius': 28,
                    'processing_duration_min': 90,
                    'machine_speed_rpm': s,
                    'raw_material_grade': g,
                    'defect_rate_pct': 5,
                }
                X_row = pd.DataFrame([row])
                y_hat = float(best_model.predict(X_row)[0])
                yield_kg = b * y_hat / 100
                bottleneck = s > 900 and m > 18
                efficiency = min(100, y_hat * (1 - row['defect_rate_pct']/100))
                lookup.append({**row, 'yield_pct': round(y_hat, 2),
                               'yield_kg': round(yield_kg, 1),
                               'bottleneck': bottleneck,
                               'efficiency_score': round(efficiency, 1)})

json_path = DATA_DIR / 'yield_lookup.json'
with open(json_path, 'w') as f:
    json.dump(lookup, f, indent=2)
print(f'Saved lookup: {json_path} ({len(lookup)} scenarios)')

In [ ]:
# CELL 10 — SUMMARY
print('=' * 55)
print('  AGROSIGHT YIELD MODEL — SUMMARY')
print('=' * 55)
print(metrics_df.round(4).to_string())
print(f'\n  Winner     : {BEST}')
print(f'  Model file : models/agrosight_yield_model.pkl')
print(f'  Simulator  : dashboard/data/yield_lookup.json')
print(f'  Charts     : dashboard/assets/04–06_*.png')
print('=' * 55)